# KV Cache Compression — Results Analysis

This notebook loads the JSON results produced by `scripts/eval_all.sh` and
generates the plots and tables used in the report.

**Run `scripts/eval_all.sh` first**, then open this notebook.

In [ ]:
import json
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')
RESULTS_DIR = pathlib.Path('../results')

## 1. Load all per-task result files

In [ ]:
records = []
for f in sorted(RESULTS_DIR.glob('*.json')):
    if f.name.startswith('summary') or f.name == 'benchmark.json':
        continue
    data = json.loads(f.read_text())
    records.append({
        'task':    data['dataset'],
        'method':  data['method'],
        'score':   data['score'],
        'latency': data['avg_latency_sec'],
        'metric':  data['metric'],
    })

df = pd.DataFrame(records)
print(df.shape)
df.head(10)

## 2. Pivot table: score by task × method

In [ ]:
pivot = df.pivot_table(index='task', columns='method', values='score')

# Add relative degradation vs full attention
if 'full' in pivot.columns:
    for m in [c for c in pivot.columns if c != 'full']:
        pivot[f'{m}_delta'] = pivot[m] - pivot['full']

pivot.round(2)

## 3. Bar chart: per-task scores

In [ ]:
method_order = ['full', 'h2o', 'streaming_llm']
method_labels = {'full': 'Full Attention', 'h2o': 'H2O (20%)', 'streaming_llm': 'StreamingLLM (20%)'}

tasks = df['task'].unique()
x = np.arange(len(tasks))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
colors = sns.color_palette('muted', n_colors=3)

for i, method in enumerate(method_order):
    sub = df[df['method'] == method].set_index('task')
    scores = [sub.loc[t, 'score'] if t in sub.index else 0 for t in tasks]
    bars = ax.bar(x + i * width, scores, width, label=method_labels.get(method, method), color=colors[i])

ax.set_xlabel('Task')
ax.set_ylabel('Score (%)')
ax.set_title('LongBench Scores: Full Attention vs KV Compression (20% budget)')
ax.set_xticks(x + width)
ax.set_xticklabels(tasks, rotation=30, ha='right')
ax.legend()
ax.set_ylim(0, 105)
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'scores_by_task.pdf', bbox_inches='tight')
plt.show()

## 4. Score vs. cache budget (efficiency–accuracy trade-off)

In [ ]:
# Parse budget from method name if you used naming convention like 'h2o_20pct'
# Adapt the logic below to match your actual result filenames.

budget_records = []
for f in sorted(RESULTS_DIR.glob('summary_*.json')):
    data = json.loads(f.read_text())
    method = data['method']
    macro  = data['macro_avg_score']
    budget_records.append({'method': method, 'macro_score': macro})

bdf = pd.DataFrame(budget_records)
bdf

## 5. Speed benchmark

In [ ]:
bench_path = RESULTS_DIR / 'benchmark.json'
if bench_path.exists():
    bench = pd.DataFrame(json.loads(bench_path.read_text()))
    display(bench)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    for method, grp in bench.groupby('method'):
        axes[0].plot(grp['seq_len'], grp['avg_tps'], marker='o', label=method)
        axes[1].plot(grp['seq_len'], grp['peak_mem_mb'], marker='o', label=method)

    axes[0].set_xlabel('Sequence length (tokens)')
    axes[0].set_ylabel('Tokens / second')
    axes[0].set_title('Throughput vs Sequence Length')
    axes[0].legend()

    axes[1].set_xlabel('Sequence length (tokens)')
    axes[1].set_ylabel('Peak GPU memory (MB)')
    axes[1].set_title('Memory vs Sequence Length')
    axes[1].legend()

    fig.tight_layout()
    fig.savefig(RESULTS_DIR / 'benchmark.pdf', bbox_inches='tight')
    plt.show()
else:
    print('Run run_benchmark.py first.')

## 6. Summary table for report

In [ ]:
summary_rows = []
for method in df['method'].unique():
    sub = df[df['method'] == method]
    summary_rows.append({
        'Method':          method,
        'Macro-avg score': round(sub['score'].mean(), 2),
        'Avg latency (s)': round(sub['latency'].mean(), 3),
        'Best task':       sub.loc[sub['score'].idxmax(), 'task'],
        'Worst task':      sub.loc[sub['score'].idxmin(), 'task'],
    })

pd.DataFrame(summary_rows).set_index('Method')